In [76]:
import os
import pandas as pd
import polars as pl
import numpy as np
import jax.numpy as jnp
import DENV_transmission_model

In [40]:
# analysis startdate
start_year = 1998
start_month = 9
end_year = 2008
assert start_year >= 1998, "earliest start_year is 1998."

# modeled municipalities
included_munis = [2611606, 2927408] # Recife and Salvador
age_groups = [(0, 1), (1, 6), (6, 12), (12, 18), (18, 25), (25,35), (35,55), (55,120)]
breaks = [end for _, end in age_groups[:-1]]
labels = [f"[{start}, {end})" for start, end in age_groups]

## Prepare the input data

In [60]:
# Compute population in start_year (n_munis x n_age_groups)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

demo = (
    pl.scan_parquet('../../data/interim/demographics/population_mun-age_1998-2026.parquet')
    .filter(pl.col("CD_MUN").is_in(included_munis))
    .filter(pl.col("year") == start_year)
    .with_columns(pl.col("age").cut(breaks=breaks, labels=labels, left_closed=True).alias("age_group"))
    .group_by([ "CD_MUN", "age_group"]).agg(pl.col("population").sum().alias("population"))
    .sort([ "CD_MUN", "age_group"])
    .collect()
)

# convert to jnp array
demo = demo.pivot(
    values="population",
    index="CD_MUN",
    on="age_group",
    aggregate_function="sum",
)
demo = jnp.array(demo.to_numpy())

In [80]:
# Compute the aging matrix (n_age_groups x n_age_groups)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# compute widths in months
widths_months = np.array([(b[1] - b[0]) * 12 for b in age_groups])

# compute aging rates (1 / width)
aging_rates = np.zeros(len(age_groups))
aging_rates[:-1] = 1.0 / widths_months[:-1]

# build the transition matrix (n_groups x n_groups)
A_aging = np.zeros((len(age_groups), len(age_groups)))
np.fill_diagonal(A_aging, -aging_rates) # negative outflows on the diagonal
for i in range(len(age_groups) - 1):
    A_aging[i + 1, i] = aging_rates[i] # positive inflows on the sub-diagonal

# convert to jnp array
A_aging = jnp.array(A_aging)

In [114]:
# Compute birth and death rates (n_months x n_munis x n_age_groups)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# get raw data (yearly)
births = pd.read_csv('../../data/interim/demographics/births_mun_1998-2026.csv')
deaths = pd.read_csv('../../data/interim/demographics/deaths_mun_1998-2026.csv')
bd = births.merge(deaths, on=['CD_MUN', 'year', 'population'], how='left')
bd = bd[bd['CD_MUN'].isin(included_munis)]

# convert data to monthly data
bd['births'] /= 12
bd['death_rate'] = (bd['deaths'] / 12) / bd['population']

# expand to monthly timestep of the model
months = pd.date_range(start=f"{start_year}-{start_month:02d}-01", end=f"{end_year}-12-31", freq="ME")
monthly_template = pd.MultiIndex.from_product([included_munis, months], names=["CD_MUN", "date"]).to_frame(index=False)
monthly_template["year"] = monthly_template["date"].dt.year
bd_monthly = pd.merge(monthly_template, bd, on=["CD_MUN", "year"], how="left")

# births (all in first age group)
births = np.zeros(shape=(len(months), len(included_munis), len(age_groups)))
births_2d = bd_monthly.pivot(index="date", columns="CD_MUN", values="births").to_numpy()
births[:,:,0] = births_2d
births = jnp.array(births)

# deaths (uniform death rate over age groups)
deaths_2d = bd_monthly.pivot(index="date", columns="CD_MUN", values="death_rate").to_numpy()
deaths = np.repeat(deaths_2d[:,:,None], len(age_groups), axis=-1)

## Load the transition matrices